# Python para IA — Introducción práctica

Repaso de los patrones de Python más usados en proyectos de IA:
variables y tipos, funciones, clases, comprehensions, manejo de errores y async básico.

## 1. Tipos de datos más usados en IA

In [ ]:
# Tipos primitivos
temperatura: float = 0.7          # parámetro de LLM
max_tokens: int = 1024
streaming: bool = True
modelo: str = 'claude-haiku-4-5-20251001'

# Listas — para mensajes, embeddings, resultados
mensajes: list[dict] = [
    {'role': 'user', 'content': 'Hola'},
    {'role': 'assistant', 'content': 'Hola, ¿en qué te ayudo?'},
]

# Diccionarios — payload de API, metadatos
config: dict = {
    'model': modelo,
    'max_tokens': max_tokens,
    'temperature': temperatura,
    'stream': streaming,
}

# None — valor por defecto para parámetros opcionales
system_prompt: str | None = None

print('Config:', config)
print('Mensajes:', len(mensajes), 'turnos')
print('System prompt:', system_prompt)

## 2. Funciones con type hints

In [ ]:
def construir_prompt(tarea: str, contexto: str = '', ejemplos: list[str] | None = None) -> str:
    partes = [tarea]
    if contexto:
        partes.append(f'Contexto: {contexto}')
    if ejemplos:
        partes.append('Ejemplos:')
        partes.extend(f'- {e}' for e in ejemplos)
    return '\n\n'.join(partes)

def calcular_coste(tokens_entrada: int, tokens_salida: int, modelo: str = 'haiku') -> float:
    precios = {
        'haiku':  {'entrada': 0.80, 'salida': 4.00},
        'sonnet': {'entrada': 3.00, 'salida': 15.00},
        'opus':   {'entrada': 15.00, 'salida': 75.00},
    }
    p = precios.get(modelo, precios['haiku'])
    return (tokens_entrada * p['entrada'] + tokens_salida * p['salida']) / 1_000_000

prompt = construir_prompt(
    tarea='Clasifica el sentimiento del texto.',
    contexto='Reseñas de productos e-commerce',
    ejemplos=['"Me encantó" → positivo', '"No funcionó" → negativo'],
)
print(prompt)
print(f'\nCoste estimado: ${calcular_coste(500, 50, "haiku"):.6f}')

## 3. Clases para encapsular lógica de IA

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class Mensaje:
    role: str
    content: str
    timestamp: datetime = field(default_factory=datetime.now)

    def to_api(self) -> dict:
        return {'role': self.role, 'content': self.content}

class Conversacion:
    def __init__(self, system: str = ''):
        self.system = system
        self.mensajes: list[Mensaje] = []

    def agregar(self, role: str, content: str) -> None:
        self.mensajes.append(Mensaje(role=role, content=content))

    def para_api(self) -> list[dict]:
        return [m.to_api() for m in self.mensajes]

    def tokens_estimados(self) -> int:
        total = sum(len(m.content.split()) for m in self.mensajes)
        return int(total * 1.3)  # aprox 1.3 tokens por palabra

    def __repr__(self) -> str:
        return f'Conversacion({len(self.mensajes)} mensajes, ~{self.tokens_estimados()} tokens)'

conv = Conversacion(system='Eres un asistente de soporte técnico.')
conv.agregar('user', 'No puedo instalar el paquete.')
conv.agregar('assistant', '¿Qué error ves exactamente?')
conv.agregar('user', 'ModuleNotFoundError: No module named anthropic')

print(conv)
print('Para API:', conv.para_api())

## 4. Comprehensions y generadores

In [ ]:
textos = [
    'El producto llegó tarde pero en buen estado.',
    'EXCELENTE CALIDAD, lo recomiendo!!!',
    'No funciona. Pésimo servicio al cliente.',
    '   Cumple con lo esperado.   ',
    '',
]

# List comprehension — limpiar y filtrar
limpios = [t.strip().lower() for t in textos if t.strip()]
print('Limpios:', limpios)

# Dict comprehension — crear índice
indice = {i: texto for i, texto in enumerate(limpios)}
print('\nÍndice:', indice)

# Generador — procesar uno a uno sin cargar todo en memoria (útil con embeddings)
def lotes(items: list, tamaño: int):
    for i in range(0, len(items), tamaño):
        yield items[i:i + tamaño]

for lote in lotes(limpios, 2):
    print(f'Lote de {len(lote)}: {lote}')

## 5. Manejo de errores con APIs

In [ ]:
import time
import random

class ErrorAPI(Exception):
    def __init__(self, mensaje: str, codigo: int = 0):
        super().__init__(mensaje)
        self.codigo = codigo

def llamar_api_con_reintentos(fn, max_reintentos: int = 3, espera_inicial: float = 1.0):
    for intento in range(max_reintentos):
        try:
            return fn()
        except ErrorAPI as e:
            if e.codigo == 429:  # rate limit
                espera = espera_inicial * (2 ** intento) + random.random()
                print(f'Rate limit. Esperando {espera:.1f}s (intento {intento + 1}/{max_reintentos})')
                time.sleep(espera)
            elif e.codigo >= 500:  # error servidor
                print(f'Error servidor {e.codigo}. Reintentando...')
                time.sleep(espera_inicial * (intento + 1))
            else:
                raise  # errores de cliente (400, 401, 403) → no reintentar
    raise ErrorAPI('Máximo de reintentos alcanzado', 429)

# Simulación
contador = {'llamadas': 0}

def api_inestable():
    contador['llamadas'] += 1
    if contador['llamadas'] < 3:
        raise ErrorAPI('Too Many Requests', 429)
    return 'OK'

resultado = llamar_api_con_reintentos(api_inestable)
print(f'Resultado: {resultado} (tras {contador["llamadas"]} llamadas)')

## 6. Async — llamadas concurrentes a la API

In [ ]:
import asyncio

async def procesar_texto(texto: str, delay: float = 0.1) -> dict:
    """Simula una llamada async a Claude."""
    await asyncio.sleep(delay)  # simula latencia de red
    palabras = len(texto.split())
    return {'texto': texto[:30] + '...', 'palabras': palabras, 'procesado': True}

async def procesar_lote(textos: list[str]) -> list[dict]:
    tareas = [procesar_texto(t) for t in textos]
    return await asyncio.gather(*tareas)  # concurrente, no secuencial

# En Jupyter se puede llamar directamente con await
textos_muestra = [
    'Primer texto para analizar con IA.',
    'Segundo texto con sentimiento positivo.',
    'Tercer texto para clasificación automática.',
]

import time
inicio = time.time()
resultados = await procesar_lote(textos_muestra)
print(f'Procesados en {time.time() - inicio:.2f}s (concurrente vs ~{0.1 * len(textos_muestra):.1f}s secuencial)')
for r in resultados:
    print(f'  {r}')

## 7. Variables de entorno y configuración

In [ ]:
import os
from pathlib import Path

# Cargar .env si existe (python-dotenv)
try:
    from dotenv import load_dotenv
    load_dotenv(Path.home() / '.env')  # o Path('.env')
    print('Variables de entorno cargadas desde .env')
except ImportError:
    print('python-dotenv no instalado: pip install python-dotenv')

# Leer claves de forma segura
api_key = os.getenv('ANTHROPIC_API_KEY', '')
if api_key:
    print(f'ANTHROPIC_API_KEY: {api_key[:8]}...{api_key[-4:]} ({len(api_key)} chars)')
else:
    print('ANTHROPIC_API_KEY no configurada — configúrala antes de hacer llamadas reales')

# Patrón recomendado: clase de configuración
from dataclasses import dataclass

@dataclass
class Config:
    anthropic_key: str = ''
    openai_key: str = ''
    modelo_defecto: str = 'claude-haiku-4-5-20251001'
    max_tokens: int = 1024
    debug: bool = False

    @classmethod
    def desde_entorno(cls) -> 'Config':
        return cls(
            anthropic_key=os.getenv('ANTHROPIC_API_KEY', ''),
            openai_key=os.getenv('OPENAI_API_KEY', ''),
            debug=os.getenv('DEBUG', '').lower() == 'true',
        )

cfg = Config.desde_entorno()
print(f'Config: modelo={cfg.modelo_defecto}, debug={cfg.debug}')